In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd

from mothernet.utils import load_ihdp_data, generate_data

In [ ]:

data, exclude_cols, y_col_name = load_ihdp_data(ihdp_path = Path("/Users/vzuev/Documents/git/git_other/CEVAE/datasets/IHDP"))
data

In [ ]:
from typing import Callable
import matplotlib.pyplot as plt
import numpy as np

from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

from mothernet.prediction.mothernet_additive import MotherNetAdditiveRegressor
from mothernet.utils import get_mn_model


def fit_mothernet_regression(x: pd.DataFrame, y: np.array) -> Callable[[pd.DataFrame], np.array]:
    train_y = pd.DataFrame(y)

    model_path = get_mn_model("baam_Daverage_l1e-05_maxnumclasses0_nsamples500_numfeatures10_yencoderlinear_05_08_2024_03_04_01_epoch_40.cpkt")
    reg = MotherNetAdditiveRegressor(device="cpu", path=model_path)

    prep = make_column_transformer((OrdinalEncoder(), x.dtypes == "category"), remainder='passthrough')
    train_x_pre = prep.fit_transform(x)
    ss = StandardScaler().fit(train_y)
    reg.fit(train_x_pre, ss.transform(train_y))  # ~14.2 sec on CPU for 747 data points
    
    def predict(test_x: pd.DataFrame) -> np.array:
        y_pred = reg.predict(prep.transform(test_x))
        return ss.inverse_transform(y_pred.reshape(-1, 1))
    return predict

def fit_rf_regression(x: pd.DataFrame, y: np.array) -> tuple[np.array, np.array]:
    rf_model = RandomForestRegressor(random_state=0)
    rf_model.fit(x, y)
    return lambda test_x: rf_model.predict(test_x)

def eval(y_ground: np.array, y_pred: np.array, title: str) -> None:
    plt.figure()
    plt.plot(y_ground, y_pred, 'o')
    plt.xlabel("y_test")
    plt.ylabel("y_pred")
    plt.title(f"{title} (MSE {mean_squared_error(y_ground, y_pred):.3f})")
    

# S-learner

In [ ]:
data_x, data_y = data.drop(columns=[*exclude_cols, y_col_name]), data[y_col_name]
train_x, test_x, train_y, test_y = train_test_split(data_x, data_y, random_state=0)

In [ ]:
for fit_regressor, title in (
    (fit_mothernet_regression, "MotherNet"),
    (fit_rf_regression, "Random Forest"),
):
    test_predict = fit_regressor(train_x, train_y)(test_x)
    eval(test_y, test_predict, f"S-learner: {title}")


# T-learner

In [ ]:
y_fact_colname, y_cfact_colname = "y_factual", "y_cfactual"
data_y_fact, data_treatment = data[y_fact_colname], data["treatment"]

def split_treated_non_treated(x: pd.DataFrame, treatment: np.array, y_fact: np.array) -> tuple[pd.DataFrame, np.array, pd.DataFrame, np.array]:
    treated = treatment == 1
    x_treated, y_treated = x.loc[treated], y_fact.loc[treated]
    x_non_treated, y_non_treated = x.loc[~treated], y_fact.loc[~treated]
    return x_treated, y_treated, x_non_treated, y_non_treated
    

for fit_regression, title in (
    (fit_mothernet_regression, "MotherNet"),
    (fit_rf_regression, "Random Forest"),
    ):
    x_train, x_test, \
    y_fact_train, _, \
    _, y_test, \
    treatment_train, treatment_test = train_test_split(
        data_x, data_y_fact, data_y, data_treatment, random_state=0)
    
    x_train_treated, y_train_treated, x_train_non_treated, y_train_non_treated = split_treated_non_treated(
        x_train,
        treatment=treatment_train, 
        y_fact=y_fact_train)
    
    
    treated_predictor = fit_regression(x_train_treated, y_train_treated)
    non_treated_predictor = fit_regression(x_train_non_treated, y_train_non_treated)

    test_pred_treated = treated_predictor(x_test)
    test_pred_non_treated = non_treated_predictor(x_test)
    y_test_pred = test_pred_treated - test_pred_non_treated
    eval(y_test, y_test_pred, title=f"T-learner: {title}")